# modular-diffusion: a block reused across stage trees

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.7.2/colab/reuse.ipynb)

A stage tree is one shared training run that is forked into one copy per stretch of the noise level; after the fork every copy trains alone on its own stretch (`geolip.modular_diffusion.experiments.stages`). A class the tree never saw can be added by one class-gated block per stage copy, trained alone with everything else frozen. This notebook measures how far such a block reaches once it is made: one block trained once on the same run left unforked and then used by every stage copy; a block from another run's tree as the starting point of a short re-fit, against a block trained fresh for as many steps; how the block and one row of the class table grow with their training length; and what the block keeps when it is placed untrained on a tree whose shared run was trained further or forked earlier.

1. install at a release tag and print the environment;
2. the data with its content hash, a smoke run of the picture score and of the paths this list uses, and the size of the job list;
3. the job list; each finished job is shipped to the Hub at once; a progress line a minute; the summary; then the machine is given back.

Every model the list stands on is trained by the list itself, so the numbers are comparable on any card type. A stopped session resumes by running the cells again: finished jobs keep their markers under `OUT`, and on a new machine the jobs already on the Hub are fetched back first.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[data,hub,judge,guard] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.7.2"
import os, sys, time, json, shutil, threading, subprocess
import torch, geolip.modular_diffusion as MD
from geolip.modular_diffusion import environment
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 7, 2), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
environment.check(None)
print("card:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (choose a GPU runtime)", "| cpus:", os.cpu_count())
OUT = "/content/reuse"               # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The data comes from the Hub as parquet files and becomes one npz of uint8 arrays with a content hash. The smoke run trains the reference for 60 steps and scores it (the score is checked against a second implementation and the class reader on the test pictures), then runs every new path for a few steps and checks its exactness keys, and times a training step on this card.

In [ ]:
# C2 - the data, a smoke run, the time of a step on this card, the job list's size
from huggingface_hub import snapshot_download
from geolip.modular_diffusion import data, score, linker
from geolip.modular_diffusion.beds import PicturesBed
from geolip.modular_diffusion.experiments import content, stages
NPZ = f"{OUT}/data/cifar10_32.npz"
REF = f"{OUT}/data/reference_inception.npz"     # the reference statistics: computed once here, read by every job
if not os.path.exists(NPZ):
    snapshot_download(repo_id="uoft-cs/cifar10", repo_type="dataset", allow_patterns=["plain_text/*.parquet"], local_dir=f"{OUT}/parquet")
    data.cifar_npz(parquet_dir=f"{OUT}/parquet/plain_text", out=NPZ)
EXPECTED_CONTENT_HASH = "be50ca799cbe6795ff13bca750f53f16c974b9f4903082fc06031b9719700016"
got = data.npz_content_hash(NPZ)
print("content hash", got, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or got == EXPECTED_CONTENT_HASH, "the npz is not the expected data"

REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" in C3 = nothing shipped)

SMOKE = f"{OUT}/smoke/plain"
smoke = content.main(variant="plain", seed=0, data=NPZ, out=SMOKE, device="cuda", scale=0.01, score=1, score_n=5000, ref=REF, reader=1)
assert "score" in smoke["score"], smoke["score"]
READER_CHECK = smoke["score"].get("reader_check")     # written by the job that computes the reference file: this one
print("class reader on the test pictures:", READER_CHECK, "| on the smoke model's samples:", (smoke["score"].get("class_read") or {}).get("accuracy"))
from torchmetrics.image.fid import FrechetInceptionDistance
bed = PicturesBed(NPZ, "cuda")
model = linker.link(f"{SMOKE}/model", device="cuda")
fake = score.sample_pictures(model, bed, 5000, steps=50, batch=1000, device="cuda")
real = bed.train_pictures()
fid = FrechetInceptionDistance(feature=2048, antialias=False).to("cuda")     # the same resize as the library's judge
for i in range(0, real.shape[0], 1000):
    fid.update(score.to_pictures(real[i:i + 1000], bed).to("cuda"), real=True)
for i in range(0, fake.shape[0], 1000):
    fid.update(fake[i:i + 1000].to("cuda"), real=False)
other = float(fid.compute())
mine = smoke["score"]["score"]
print(f"picture score: library {mine:.4f} | torchmetrics, the same resize {other:.4f} (relative difference {abs(mine - other) / other:.2e}) (5,000 pictures)")
del model, bed, real, fake, fid
torch.cuda.empty_cache()

# the paths this list uses for a few steps each, with det=on as the job list runs them
S = f"{OUT}/smoke"
common = dict(seed=0, data=NPZ, device="cuda", scale=0.01, det="on", score=0)
k3 = dict(cuts="0.25,0.65")
stages.main(variant="source", out=f"{S}/nsrc", without_classes="9", **common)                      # a source without class 9
stages.main(variant="copies", out=f"{S}/ntree", source=f"{S}/nsrc", **k3, **common)               # its three copies
stages.main(variant="copies", out=f"{S}/none", source=f"{S}/nsrc", **common)                       # the same run with no fork: a tree of one
stages.main(variant="source", out=f"{S}/nlong", without_classes="9", total=6000, **common)         # a longer source and its three copies
stages.main(variant="copies", out=f"{S}/nlong_tree", source=f"{S}/nlong", **k3, **common)
block = dict(parts="expert:9", **common)
SM = {}
SM["a block on the tree"] = stages.main(variant="parts", out=f"{S}/add_block", base=f"{S}/ntree", only_classes="9", total=1500, **block)
SM["a block on the tree of one"] = stages.main(variant="parts", out=f"{S}/add_block_one", base=f"{S}/none", only_classes="9", total=1500, **block)
SM["that block for every stage"] = stages.main(variant="parts", out=f"{S}/block_one_on_tree", base=f"{S}/ntree", init_from=f"{S}/add_block_one", init_stage=0, total=0, **block)
SM["the block on a second tree"] = stages.main(variant="parts", out=f"{S}/block_on_long", base=f"{S}/nlong_tree", init_from=f"{S}/add_block", total=0, **block)
SM["the block re-fit there"] = stages.main(variant="parts", out=f"{S}/block_refit", base=f"{S}/nlong_tree", init_from=f"{S}/add_block", only_classes="9", total=300, **block)
SM["a row on the tree of one"] = stages.main(variant="parts", out=f"{S}/add_row_one", base=f"{S}/none", parts="row:9", only_classes="9", total=1500, **common)
SM["that row for every stage"] = stages.main(variant="parts", out=f"{S}/row_one_on_tree", base=f"{S}/ntree", parts="row:9", init_from=f"{S}/add_row_one", init_stage=0, total=0, **common)
for name, r in SM.items():
    assert r["as_source"]["equals_stored"], f"{name}: the base read again is not its stored read"
    assert r["exact"]["others_max_abs"] == 0.0, f"{name}: rows the new parts are not on have changed"
    assert r["parts_off"]["equals_base"], f"{name}: the tree without the new parts is not the base"
g8 = lambda r: r["averaged"]["g8"]["loss_over_levels"]
print("smoke of the new paths (loss on the g8 average):", {k: round(g8(r), 4) for k, r in SM.items()})
timing = stages.main(variant="source", seed=0, data=NPZ, out=f"{S}/timing", device="cuda", total=600, det="on", score=0, sheet=0)
RATE = 1.0 / timing["parts"][0]["seconds_per_step"]         # the reference model's steps a second on this card, det=on
print(f"this card ({torch.cuda.get_device_name(0)}): {RATE:.1f} reference-model steps a second with det=on")
INSTRUMENTS = {"reader_check": READER_CHECK, "smoke_class_read": smoke["score"].get("class_read"),
               "score_check": {"library": mine, "second_same_resize": other, "pictures": 5000},
               "card": torch.cuda.get_device_name(0), "steps_a_second": RATE}

# the job list: (tag, experiment, arguments, the jobs it waits for, its cost in reference-model steps); every model it
# stands on is trained by the list itself
SCORE = 1700                                   # one scoring of 10,000 pictures, in reference-model steps
RUNS = f"{OUT}/runs"
TREE = "score_last=0 score_avg=tail100"        # a tree is judged and scored on the average of its last 100 steps
K3 = "cuts=0.25,0.65"
nine = "without_classes=9"
BASES, ONE, FOREIGN, DOSE, FURTHER = [], [], [], [], []
for s in (0, 1):
    o = 1 - s
    run = lambda t, seed=s: f"{RUNS}/{t}_s{seed}"
    # the tree that never saw one class: the shared run trained without class 9 and forked into three copies, the same
    # run with no fork (a tree of one), and the block of record: one block per stage copy for class 9, trained alone
    BASES += [(f"nine_src_s{s}", "stages", f"variant=source seed={s} total=4000 {nine} score=0", [], 4000),
              (f"nine_tree_s{s}", "stages", f"variant=copies seed={s} {K3} total=2000 source={run('nine_src')} {TREE}", [f"nine_src_s{s}"], 2000 + SCORE),
              (f"nine_one_s{s}", "stages", f"variant=copies seed={s} total=2000 source={run('nine_src')} {TREE}", [f"nine_src_s{s}"], 2000 + SCORE),
              (f"add9_block_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 only_classes=9 total=1500 base={run('nine_tree')}",
               [f"nine_tree_s{s}"], 700 + SCORE)]
    # one block trained once for every stage: trained on the tree of one, then used untrained by all three stage copies
    ONE += [(f"add9_block_one_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 only_classes=9 total=1500 base={run('nine_one')}",
             [f"nine_one_s{s}"], 700 + SCORE),
            (f"block_one_on_tree_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 base={run('nine_tree')} init_from={run('add9_block_one')} init_stage=0 total=0",
             [f"nine_tree_s{s}", f"add9_block_one_s{s}"], 200 + SCORE)]
    # a block from another run as a starting point: a fresh block trained 300 steps on this tree, and this seed's block
    # re-fit for 300 steps on the other seed's tree
    FOREIGN += [(f"fresh300_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 only_classes=9 total=300 base={run('nine_tree')}",
                 [f"nine_tree_s{s}"], 330 + SCORE),
                (f"block_refit_other_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 base={run('nine_tree', o)} init_from={run('add9_block')} only_classes=9 total=300",
                 [f"nine_tree_s{o}", f"add9_block_s{s}"], 330 + SCORE)]
    # how a piece grows with its training length: the block at 500 and 4,500 steps (300 and 1,500 are above); one row of
    # the class table trained 6,000 steps on the tree of one, then used untrained by every stage copy of the tree
    DOSE += [(f"add9_block_500_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 only_classes=9 total=500 base={run('nine_tree')}",
              [f"nine_tree_s{s}"], 250 + SCORE),
             (f"add9_block_4500_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 only_classes=9 total=4500 base={run('nine_tree')}",
              [f"nine_tree_s{s}"], 2000 + SCORE),
             (f"add9_row_one_6000_s{s}", "stages", f"variant=parts seed={s} parts=row:9 only_classes=9 total=6000 base={run('nine_one')}",
              [f"nine_one_s{s}"], 4200 + SCORE),
             (f"row_one_6000_on_tree_s{s}", "stages", f"variant=parts seed={s} parts=row:9 base={run('nine_tree')} init_from={run('add9_row_one_6000')} init_stage=0 total=0",
              [f"nine_tree_s{s}", f"add9_row_one_6000_s{s}"], 200 + SCORE)]
    # the shared run trained further (to 5,000 or 6,000 steps) or forked earlier (at 2,000), each forked into three
    # copies, with the block of record placed on it untrained
    for at, copies in ((5000, 1000), (6000, 2000), (2000, 4000)):
        FURTHER += [(f"nine_src_{at}_s{s}", "stages", f"variant=source seed={s} total={at} {nine} score=0", [], at),
                    (f"nine_tree_f{at}_s{s}", "stages", f"variant=copies seed={s} {K3} total={copies} source={run(f'nine_src_{at}')} {TREE}",
                     [f"nine_src_{at}_s{s}"], copies + SCORE),
                    (f"block_on_f{at}_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 base={run(f'nine_tree_f{at}')} init_from={run('add9_block')} total=0",
                     [f"nine_tree_f{at}_s{s}", f"add9_block_s{s}"], 200 + SCORE)]
    # on the tree of the run trained to 6,000 steps: the block re-fit for 300 steps, and a block trained there from birth
    FURTHER += [(f"block_refit_f6000_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 base={run('nine_tree_f6000')} init_from={run('add9_block')} only_classes=9 total=300",
                 [f"nine_tree_f6000_s{s}", f"add9_block_s{s}"], 330 + SCORE),
                (f"add9_block_f6000_s{s}", "stages", f"variant=parts seed={s} parts=expert:9 only_classes=9 total=1500 base={run('nine_tree_f6000')}",
                 [f"nine_tree_f6000_s{s}"], 700 + SCORE)]
ALL = BASES + FURTHER + ONE + FOREIGN + DOSE       # in the order of what is asked first
assert len({t for t, *_ in ALL}) == len(ALL), "a job tag is listed twice"
assert all(n in {t for t, *_ in ALL} for _, _, _, needs, _ in ALL for n in needs), "a job waits for a job that is not in the list"
STEPS = {t: c + 900 for t, _, _, _, c in ALL}      # a job's time in reference-model steps (its training, its reads, its scoring)
total = sum(STEPS.values())
first = sum(STEPS[t] for t, *_ in BASES + FURTHER)
print(f"the job list: {len(ALL)} jobs, {total:,.0f} reference-model steps in all: about {total / RATE / 60:.0f} min on this card with one runner "
      f"(the first {len(BASES + FURTHER)} jobs, the tree without one class, its block, and the block placed on trees whose shared run was trained further or forked earlier: about {first / RATE / 60:.0f} min)")

The runners work the job list in order, every job with deterministic algorithms switched on (`det=on`), so a job repeats to the last digit whatever runs beside it. Each finished job folder is shipped to `REPO` under its experiment's own folder with the token from the notebook's secret `HF_TOKEN` (placed in this process's environment only; the jobs themselves run without it). When the list has ended and its report is shipped, the machine is given back (`RELEASE`), since an idle machine keeps spending.

In [ ]:
# C3 - the runners, shipping after each job, a progress line a minute, the summary, the machine given back
from geolip.modular_diffusion import jobs, hub
PREFIX = {"stages": "modular_diffusion/reuse"}
RELEASE = True                              # give the machine back when the list has ended and the report is shipped
RUNNERS = 1                                  # how many jobs run at once (each claims its job; more pay only where one job leaves the card idle)
EXTRA = ""                                   # more arguments for every training job (empty in a real run)
EXTRA_READS = ""                             # more arguments for every chain read (empty in a real run)
kind = {}

def line(tag, exp, args, needs):
    kind[tag] = exp
    a = " " + args
    a += "" if " det=" in a else " det=on"
    a += "" if " device=" in a else " device=cuda threads=4"
    a += "" if " score=" in a else " score=1"
    a += " reader=1" if exp == "stages" and " score=0" not in a else ""      # the class read needs a scored job
    extra = EXTRA_READS if exp == "chain_reads" else EXTRA
    return f"{tag} | 60 | geolip.modular_diffusion.run | {exp}{a} data={NPZ} ref={REF}{extra}" + (f" | needs={','.join(needs)}" if needs else "")

tags = [t for t, *_ in ALL]
roster = f"{OUT}/roster.txt"
with open(roster, "w", encoding="utf-8", newline="\n") as fh:
    fh.write("\n".join(line(t, e, a, n) for t, e, a, n, _ in ALL) + "\nEND\n")
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    try:    # on a new machine: fetch this list's jobs that are already shipped and mark them done
        got_dir = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX[kind[t]]}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
        for t in tags:
            src = f"{got_dir}/{PREFIX[kind[t]]}/{t}"
            if os.path.exists(f"{src}/result.json") and not mark(t, "DONE"):
                shutil.copytree(src, f"{RUNS}/{t}", dirs_exist_ok=True)
                open(f"{RUNS}/{t}/DONE", "w").write("fetched back from the Hub\n")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
for t in tags:      # a claim left by an interrupted session would hold its job forever
    if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
        os.remove(f"{RUNS}/{t}/CLAIM")

def after_job(tag, folder, ok):
    if ok and REPO:
        hub.ship(folder, REPO, f"{PREFIX[kind[tag]]}/{tag}", f"{kind[tag].replace('_', ' ')}: {tag}")

t0 = time.time()
left0 = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
print(f"start: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; {left0:,.0f} steps to run; {RUNNERS} runner(s)")
runners = [threading.Thread(target=jobs.run_roster, daemon=True,
                            kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster_{i}.log", card_room_mib=None,
                                        ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job,
                                        env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
           for i in range(RUNNERS)]
for r in runners:
    r.start()
    time.sleep(2)
while any(r.is_alive() for r in runners):
    time.sleep(60)
    done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
    left = sum(STEPS[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    spent = (time.time() - t0) / 60
    eta = spent * left / (left0 - left) if left0 > left else float("nan")
    print(f"jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about {eta:.0f} min left", flush=True)

summary = stages.summarize(RUNS)
print(summary)
REPORT = f"{OUT}/report"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)   # the smoke's checks, the card, its speed
for f in ["roster.txt"] + [f"roster_{i}.log" for i in range(RUNNERS)]:
    if os.path.exists(f"{OUT}/{f}"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in tags:      # the logs of the jobs that failed (a job's own log files are not shipped with its folder)
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, "modular_diffusion/reuse/_report", "a block reused across stage trees: summary and roster logs") if REPO else None
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()